In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Sovereign Fixed Income Data Pipeline
# MAGIC **Orchestration Layer** using modular imports from `src/`

# COMMAND ----------

import os
import logging
from datetime import date, timedelta, datetime

# moduylar imports
from src.ingestion import bcb, tesouro
from src.curves import bootstrap  

#  cluster logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("yield_curve_pipeline")

# COMMAND ----------

# runtime parameters passed by Azure Data Factory
dbutils.widgets.text("start_date", (date.today() - timedelta(days=365*2)).strftime("%Y-%m-%d"))
dbutils.widgets.text("silver_db_name", "silver")

START_DATE_STR = dbutils.widgets.get("start_date")
SILVER_DB = dbutils.widgets.get("silver_db_name")
start_date_parsed = datetime.strptime(START_DATE_STR, "%Y-%m-%d").date()

# COMMAND ----------

# Execute the decoupled ingestion tasks
logger.info("Executing Bacen and IPCA ingestion stream...")
data_bcb = bcb.fetch_all(start_date=start_date_parsed, include_focus=False)

logger.info("Executing Tesouro Transparente nominal and real market asset stream...")
data_tesouro = tesouro.fetch_all(start_date=start_date_parsed)

# Handle VNA Fallback dynamically using  imported modules if needed
df_vna = data_bcb.get('vna_ntn_b')
if df_vna is None or df_vna.empty:
    logger.warning("Bacen VNA endpoint offline. Triggering src imputation module...")
    historical_trade_dates = data_tesouro['ntnb_zero']['date'].unique()
    
    # Impute VNA from IPCA:
    df_vna = bcb.impute_vna_from_ipca(data_bcb.get('ipca_monthly'), historical_trade_dates)

# COMMAND ----------

# 5. Write out to Managed Delta Tables
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")

spark.createDataFrame(df_vna).write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_DB}.vna_factors")
spark.createDataFrame(data_tesouro['ntnb_zero']).write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_DB}.ntnb_market_clean")

logger.info("Bronze-to-Silver data pipeline processing complete.")